# CV Deep Learning - Tugas Deep Learning

**Proyek Computer Vision untuk Analisis Rambut Salon (TIEN SALON)**

**Catatan Penting:**
- Notebook ini mengkonsolidasi preprocessing, training experiments, dan evaluation pipeline CV deep learning untuk klasifikasi jenis dan panjang rambut.
- **Tidak termasuk RAG evaluation** (RAG eval dipisahkan jika diperlukan).
- Output visualization disimpan di `apps/api/cv/outputs/`

---

## Daftar Isi

1. [Section 1: CONFIGURATION & IMPORTS](#section-1-config--imports)
2. [Section 2: PREPROCESSING PIPELINE](#section-2-preprocessing-pipeline)
3. [Section 3: TRAINING EXPERIMENTS](#section-3-training-experiments)
4. [Section 4: EVALUATION](#section-4-evaluation)

### Section 1: CONFIGURATION & IMPORTS

### Penjelasan

Bagian ini mengatur semua konfigurasi awal untuk eksperimen deep learning:
- **Seeds**: Menetapkan seed untuk reproducibility (random number generators dari PyTorch dan Python)
- **ImageNet normalization**: Normalisasi menggunakan mean=[0.485,0.456,0.406] dan std=[0.229,0.224,0.225] sesuai ImageNet statistics
- **Paths**: Path ke dataset mentah dan folder output untuk artifacts
- **Imports**: Semua library yang diperlukan (PyTorch, torchvision, PIL, dll.)

Configurasi ini penting karena memastikan eksperimen dapat direproduksi dengan hasil yang sama setiap kali dijalankan.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageOps
import random
import json
from pathlib import Path

# Configuration
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
INPUT_SIZE = 224
RESIZE_SIDE = 256
CROP = 224
MIN_SIDE = 224

# Augmentation for training
RRC_SCALE = (0.875, 1.0)
RRC_RATIO = (3.0 / 4.0, 4.0 / 3.0)
FLIP_P = 0.5
ROT_DEG = 10
AUG_REPEAT = 2

# Paths
DATASET = Path('dataset/figaro1k')
OUT = Path('outputs')

### Section 2: PREPROCESSING PIPELINE

### Penjelasan

Pipeline preprocessing terdiri dari:
- **RGB 8-bit sRGB**: Input harus RGB, bukan grayscale (warna penting untuk salon)
- **EXIF transpose**: Auto-orient sesuai EXIF orientation
- **Resize keep-aspect**: Resize sisi pendek ke 256px
- **Center crop 224**: Crop center 224x224
- **ToTensor + Normalize**: Convert ke Tensor + ImageNet normalization
- **Output**: Tensor NCHW float32 (1,3,224,224)

**Augmentasi (training only):**
- RandomResizedCrop(scale 0.875–1.0, ratio 3/4–4/3)
- RandomHorizontalFlip(p=0.5)
- RandomRotation(degrees=±10°)
- Setiap citra train → (1+2) = 3 versi
- Val/inference identik tanpa augmentasi

### Section 3: TRAINING EXPERIMENTS

### Penjelasan

Bagian ini mencakup berbagai eksperimen training backbone dan ablation studies:
- **Backbone benchmark**: Benchmark 15+ backbone architectures untuk classification task
- **Ablation study**: Variasi seeds (s1, s7, s123) untuk melihat stabilitas
- **Ensemble strategies**: Combination of multiple checkpoints dengan weighted averaging
- **Best model selection**: Memilih model terbaik berdasarkan accuracy/F1 score
- **Export**: Export best checkpoint ke ONNX format

### Section 4: EVALUATION

### Penjelasan

Bagian ini mencakup evaluasi komprehensif terhadap model terbaik:
- **Confusion matrices**: Multi-class confusion matrix heatmaps untuk hair type & hair length
- **Per-class metrics**: Bar charts untuk precision/recall/F1 per kelas
- **Separability analysis**: Boxplot F-statistics dan scatter plots untuk keriting vs sangat-keriting
- **Comparison tables**: Tabel perbandingan dengan geometric labeling dan Gemini methods
- **Summary report**: Export summary report ke JSON